In [1]:
import os
from pathlib import Path

import polars as pl

In [2]:
DATABASE_URI = os.environ["DATABASE_URI"]

In [3]:
RANDOM_SEED = 42

In [4]:
DATASETS_PATH = Path(
    "/Users/luis/projets/beta.gouv/qfdmod/quefairedemesobjets/data-platform/notebooks/deduplication/datasets"
)

# Annotation manuelle (Christian)

In [5]:
df = (
    pl.read_csv(
        "datasets/Clusterisation comme Data-inclusion - par département - deduplication_clusters_26_p_90.csv"
    )
    .lazy()
    .rename({"Good ? ": "Good ?"}, strict=False)
)
df_pairs = (
    df.join(df, on="cluster_id")
    .filter(pl.col("Good ?").is_in(["oui", "non"]))
    .with_columns(
        pl.min_horizontal(
            pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
        ).alias("identifiant_unique_i"),
        pl.max_horizontal(
            pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
        ).alias("identifiant_unique_j"),
        (pl.col("Good ?") == "oui").alias("label"),
    )
    .filter(
        (pl.col("identifiant_unique_i") != pl.col("identifiant_unique_j"))
        & (
            pl.col("cluster_id")
            .cum_count()
            .over(["cluster_id", "identifiant_unique_i", "identifiant_unique_j"])
            == 1
        )
    )
    .select(["identifiant_unique_i", "identifiant_unique_j", "label", "cluster_id"])
)

In [6]:
df_cluster_ids = (
    df_pairs.group_by("cluster_id")
    .agg(
        pl.col("identifiant_unique_i")
        .list.concat(pl.col("identifiant_unique_j"))
        .list.explode()
        .alias("ids")
    )
    .with_columns(
        pl.col("ids").list.unique().sort().hash(RANDOM_SEED).alias("cluster_id_hash")
    )
)

/var/folders/g4/t3006b6x41b4f5tmkgqprc0c0000gn/T/ipykernel_23773/1105164813.py:6: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .list.explode()


In [7]:
df_pairs.collect()

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,i64
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false,0
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false,0
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false,0
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false,0
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488016001""",false,0
…,…,…,…
"""BAL_44.343863, 5.283499""","""BAL_44.4070419, 5.1996969""",false,80
"""aliapur_250708000000010""","""reseau_vrac_reemploi_vy_44.601…",false,92
"""refashion_TLC-REFASHION-PAV-34…","""refashion_TLC-REFASHION-PAV-34…",false,94


In [8]:
df_pairs_with_cluster_id = df_pairs.join(
    df_cluster_ids, on="cluster_id", validate="m:1"
).collect()

In [9]:
df_pairs_with_cluster_id.select(
    [
        "identifiant_unique_i",
        "identifiant_unique_j",
        "cluster_id_hash",
    ]
)

identifiant_unique_i,identifiant_unique_j,cluster_id_hash
str,str,u64
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",12780274912313031330
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",12780274912313031330
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",12780274912313031330
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",12780274912313031330
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488016001""",12780274912313031330
…,…,…
"""BAL_44.343863, 5.283499""","""BAL_44.4070419, 5.1996969""",2496823660075169213
"""aliapur_250708000000010""","""reseau_vrac_reemploi_vy_44.601…",477849465000432103
"""refashion_TLC-REFASHION-PAV-34…","""refashion_TLC-REFASHION-PAV-34…",7944515879513716650


In [10]:
def create_entity_pairs_from_datasets(datasets_folder_path: Path) -> pl.DataFrame:
    csv_files_to_load = datasets_folder_path.glob("Clusterisation *.csv")

    dfs = []
    for csv_filepath in csv_files_to_load:
        df = (
            pl.read_csv(csv_filepath).lazy().rename({"Good ? ": "Good ?"}, strict=False)
        )
        df_pairs = (
            df.join(df, on="cluster_id")
            .filter(pl.col("Good ?").is_in(["oui", "non"]))
            .with_columns(
                pl.min_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_i"),
                pl.max_horizontal(
                    pl.col("identifiant_unique"), pl.col("identifiant_unique_right")
                ).alias("identifiant_unique_j"),
                (pl.col("Good ?") == "oui").alias("label"),
            )
            .filter(
                (pl.col("identifiant_unique_i") != pl.col("identifiant_unique_j"))
                & (
                    pl.col("cluster_id")
                    .cum_count()
                    .over(
                        ["cluster_id", "identifiant_unique_i", "identifiant_unique_j"]
                    )
                    == 1
                )
            )
            .select(
                ["identifiant_unique_i", "identifiant_unique_j", "label", "cluster_id"]
            )
        )

        df_cluster_ids = (
            df_pairs.filter(pl.col("label"))
            .group_by("cluster_id")
            .agg(
                pl.col("identifiant_unique_i")
                .list.concat(pl.col("identifiant_unique_j"))
                .list.explode()
                .alias("ids")
            )
            .with_columns(
                pl.col("ids")
                .list.unique()
                .sort()
                .hash(RANDOM_SEED)
                .alias("cluster_id_hash")
            )
        )
        df_pairs_with_cluster_id = df_pairs.join(
            df_cluster_ids, on=["cluster_id"], validate="m:1", how="left"
        ).with_columns(
            pl.when(pl.col("label").not_())
            .then(None)
            .otherwise(pl.col("cluster_id_hash"))
            .alias("cluster_id_hash")
        )  # Keep cluster_id only on pairs that are true

        dfs.append(
            df_pairs_with_cluster_id.select(
                [
                    "identifiant_unique_i",
                    "identifiant_unique_j",
                    "label",
                    pl.col("cluster_id_hash").alias("cluster_id").cast(pl.String),
                ]
            )
        )

    df_pairs_concat = (
        pl.concat(dfs)
        .unique(["identifiant_unique_i", "identifiant_unique_j"])
        .sort(["identifiant_unique_i", "identifiant_unique_j"])
        .collect()
    )

    return df_pairs_concat

In [11]:
df_manual_labeling = create_entity_pairs_from_datasets(DATASETS_PATH)

/var/folders/g4/t3006b6x41b4f5tmkgqprc0c0000gn/T/ipykernel_23773/3762411360.py:43: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .list.explode()


In [12]:
df_manual_labeling

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true,"""7758461304653219539"""
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false,null
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false,null
…,…,…,…
"""reseau_vrac_reemploi_14u_47.55…","""reseau_vrac_reemploi_1EK_47_-2""",true,"""3143912265457579969"""
"""reseau_vrac_reemploi_1DB_45_4""","""reseau_vrac_reemploi_tR_45.118…",true,"""4089312382647365537"""
"""reseau_vrac_reemploi_wA_44.818…","""reseau_vrac_reemploi_wC_44.900…",false,null


In [13]:
df_manual_labeling.group_by("label").len()

label,len
bool,u32
false,130
true,39


# Depuis la base

## Examples négatifs via changement de parent dont l'ancien parent existe toujours

**Vrais négatifs via changement de parent :** Acteurs qui ne sont pas des parents et dont le parent lié aujourd'hui diffère de celui trouvé dans les suggestions de clustering.

In [14]:
sql_neg_method_1 = """
with suggestions_avec_creation_de_parent as 
(
select
	ds.id,
	ds.statut,
	jsonb_path_query_array(ds.contexte, '$.fuzzy_details[*].identifiant_unique') as identifiant_unique_list,
	ds.suggestion->'changes'->0->'model_params'->>'id' as id_parent,
	ds.suggestion->'changes'->0->>'reason' as suggestion_reason,
	ds.modifie_le
from
	data_suggestion ds
inner join data_suggestioncohorte ds2 
on
	ds.suggestion_cohorte_id = ds2.id
	and ds2.type_action = 'CLUSTERING'
where
	ds.contexte->'fuzzy_details' is not null
	and ds.suggestion->'changes'->0->>'reason' in ('1️⃣ 1 seul parent existant → à garder','➕ Pas de parent → à créer')
	and ds.statut='SUCCES'
),
suggestions_explosees as (
select
	s.id,
	s.id_parent,
	s.suggestion_reason,
	s.modifie_le,
	s_lid.id_acteur
from
	suggestions_avec_creation_de_parent s,
	 jsonb_array_elements_text(s.identifiant_unique_list) as s_lid(id_acteur)
where id_acteur::text<>s.id_parent
),
suggestions_avec_parent_existant as (
SELECT 
	se.*
from suggestions_explosees se
inner join qfdmo_vueacteur qv on se.id_parent=qv.identifiant_unique
),
suggestions_vrais_negatifs as (
select
	 qv.identifiant_unique,
	 qv."uuid",
	 se.id_parent as suggestion_parent_id,
	 se.id as suggestion_id,
	 se.suggestion_reason
from
	qfdmo_vueacteur qv
inner join suggestions_avec_parent_existant se on qv.identifiant_unique = se.id_acteur and qv.parent_id!=se.id_parent
where
	not qv.est_parent
	and qv.statut <> 'SUPPRIME'
	and qv.modifie_le <= now() - interval '1 day'
)
select
	identifiant_unique as identifiant_unique_i,
	suggestion_parent_id as identifiant_unique_j
from suggestions_vrais_negatifs
union all
select
	svn.identifiant_unique as identifiant_unique_i,
	qv.identifiant_unique as identifiant_unique_j
from suggestions_vrais_negatifs svn
inner join 	qfdmo_vueacteur qv
on svn.suggestion_parent_id=qv.parent_id and svn.identifiant_unique!=qv.identifiant_unique
"""

In [15]:
df_negatives_via_parent_change = pl.read_database_uri(
    query=sql_neg_method_1, uri=DATABASE_URI
).with_columns(
    pl.lit(False).alias("label"), pl.lit(None).alias("cluster_id").cast(pl.String)
)

In [16]:
df_negatives_via_parent_change

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""ocab_valobat_DCT_38100_GREN_01""","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false,null
"""ocab_ecomaison_DCT_38100_GREN_…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false,null
"""cyclevia_decb5827-1c63-4856-9c…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false,null
"""ocab_ecominero_DCT_38100_GREN_…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false,null
"""ocab_valdelia_DCT_38100_GREN_0…","""e1b4bebc-1ff1-5a3f-b2bb-1b18bf…",false,null
…,…,…,…
"""ocab_valdelia_DCT_38100_GREN_0…","""aliapur_21141""",false,null
"""ocab_valdelia_DCT_38100_GREN_0…","""ademesinoedecheteries_4130""",false,null
"""ocab_valdelia_DCT_38100_GREN_0…","""ocab_valobat_DCT_38021_GREN_00""",false,null


# Par échantillonage aléatoire

**Vrai négatifs :** le but est de créer des paires d'acteurs qui ne sont pas reliées entre elles.
Contraintes :

- Pas de parent en commun
- Pas de relation parent<->enfant
- Pas la même source
- Distance géographique > 10km OU pas le même département

In [17]:
sql_neg_method_2 = """
WITH
  sampled AS (
    SELECT
      identifiant_unique,
      parent_id,
      source_id,
      "location",
      code_postal,
      acteur_type_id,
      row_number() OVER (
        ORDER BY
          MD5(identifiant_unique::TEXT || '42')
      ) AS rn -- Seed pour le reproductibilité
    FROM
      qfdmo_vueacteur qv
    WHERE
      qv.statut <> 'SUPPRIME'
  ),
  sampled_filtered AS (
    SELECT
      *
    FROM
      sampled
    WHERE
      rn <= 100000 -- Permet la création d'environ 50k paires
  ),
  randomized_pairs AS (
    SELECT
      identifiant_unique,
      parent_id,
      source_id,
      "location",
      code_postal,
      acteur_type_id,
      row_number() OVER (
        ORDER BY
          MD5(identifiant_unique::TEXT || '42')
      ) AS rn -- Mélange aléatoire REPRODUCTIBLE de l'échantillon
    FROM
      sampled_filtered
  )
  -- Création des paires
SELECT
  MIN(a.identifiant_unique) AS identifiant_unique_i,
  MAX(b.identifiant_unique) AS identifiant_unique_j
FROM
  randomized_pairs a
  JOIN randomized_pairs b ON FLOOR((a.rn - 1) / 2) = FLOOR((b.rn - 1) / 2) -- Groupe en paire en utilisant le ~row_number/2 comme clé
  AND a.identifiant_unique < b.identifiant_unique -- S'assure de ne pas avoir de paires avec les mêmes ids dans un ordre différent
  AND a.identifiant_unique <> coalesce(b.parent_id, '') -- acteur_j n'est pas un parent de acteur_i
  AND b.identifiant_unique <> coalesce(a.parent_id, '') -- acteur_i n'est pas un parent de acteur_j
  AND coalesce(a.parent_id, 'N/A') != coalesce(b.parent_id, 'N/A2') -- les deux acteurs n'ont pas le même parent
  AND coalesce(a.source_id, -1) != coalesce(b.source_id, -2) -- les deux acteurs n'ont pas la même source
  AND coalesce(a.acteur_type_id, -1) != coalesce(b.acteur_type_id, -2) -- les deux acteurs ne sont pas du même type
  AND a.acteur_type_id!=10 AND b.acteur_type_id!=10 -- Exclusion des PAV publics
  AND (
    ST_DistanceSpheroid (a."location"::geometry, b."location"::geometry) >= 10000
    OR left(a.code_postal, 2) != left(b.code_postal, 2)
  ) -- plus de 10km entre les deux ou département différent
GROUP BY
  FLOOR((a.rn - 1) / 2)
"""

In [18]:
df_negatives_random = pl.read_database_uri(
    query=sql_neg_method_2, uri=DATABASE_URI
).with_columns(
    pl.lit(False).alias("label"), pl.lit(None).alias("cluster_id").cast(pl.String)
)

In [19]:
df_negatives_random

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""BAL_50.6011024, 5.6316395""","""ecopae_ECOPAEPDR00430""",false,null
"""aliapur_241121000000015""","""atout_cafe_144140_reparation_0…",false,null
"""cma_reparacteur_12751""","""communaute_emmaus_bourg_en_bre…",false,null
"""casellas_louis_141006_reparati…","""ecodds_FD2432""",false,null
"""bijouterie_gael_121705_reparat…","""corepile_73-DIS-0405""",false,null
…,…,…,…
"""batribox_ENT231107SU""","""inspirations_192800_reparation""",false,null
"""cmareparacteur_68242""","""ordredespharmaciens_E02488ABC2…",false,null
"""ecodds_FD0075""","""heliolite_175214_reparation_06…",false,null


# Vrais positifs

In [32]:
sql_pos = """
with pool as 
(
select
	qv.identifiant_unique ,
	qv.parent_id,
	case
		when qv.est_parent then qv.identifiant_unique
		else qv.parent_id
	end as cleaned_parent_id -- Le but est de permettre que des parents soient dans le pool (normalement ils n'ont pas de parent_id)
from
	qfdmo_vueacteur qv),
paires_dupliquees as (
select
	least(p.identifiant_unique, p2.identifiant_unique) as identifiant_unique_i,
	greatest(p.identifiant_unique, p2.identifiant_unique) as identifiant_unique_j,
    p.cleaned_parent_id
from
	pool p
inner join pool p2 on
	p.cleaned_parent_id = p2.cleaned_parent_id
	and p.identifiant_unique != p2.identifiant_unique
)
select
	identifiant_unique_i,
	identifiant_unique_j,
    cleaned_parent_id
from
	(
	select
		*,
		row_number() over (partition by identifiant_unique_i,
		identifiant_unique_j) as rn
	from
		paires_dupliquees
)
where
	rn = 1
"""

In [33]:
df_positives = (
    pl.read_database_uri(query=sql_pos, uri=DATABASE_URI)
    .with_columns(
        pl.lit(True).alias("label"), pl.col("cleaned_parent_id").alias("cluster_id")
    )
    .select(["identifiant_unique_i", "identifiant_unique_j", "label", "cluster_id"])
)

In [34]:
df_positives

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""000082e5-9817-5374-b9e6-a42d06…","""citeo_5088173001""",true,"""000082e5-9817-5374-b9e6-a42d06…"
"""000082e5-9817-5374-b9e6-a42d06…","""citeo_5088173002""",true,"""000082e5-9817-5374-b9e6-a42d06…"
"""000245cf-f393-5e59-b2ff-a6e197…","""citeo_1764957001""",true,"""000245cf-f393-5e59-b2ff-a6e197…"
"""000245cf-f393-5e59-b2ff-a6e197…","""citeo_1764957002""",true,"""000245cf-f393-5e59-b2ff-a6e197…"
"""00032a9a-988b-5e2a-8eb0-b8b686…","""citeo_1784187001""",true,"""00032a9a-988b-5e2a-8eb0-b8b686…"
…,…,…,…
"""valdelia_836""","""valtri__histoires_sans_fin_581…",true,"""353cce06-b798-472a-9335-dd57ab…"
"""valdelia_EA_0000298""","""valdelia_PMCB_0000844""",true,"""a260a1fd-6cf5-5a21-8b6a-0663c2…"
"""valdelia_EA_0001073""","""valdelia_EA_0001077""",true,"""e2347007-e96b-43f3-8936-cf9b5c…"


# Constitution d'un dataset équilibré

Même nombre d'examples positifs et négatifs. Pour commencer 1000 de chaque en privilégiant les annotations manuelles en en dernier lieu les paires issues d'un échantillonage aléatoire

In [35]:
df_positives.sort(pl.col("cluster_id").shuffle())

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""805fa54b-403b-5d39-83e6-7ce904…","""citeo_5074427002""",true,"""805fa54b-403b-5d39-83e6-7ce904…"
"""b8b449f5-6251-583c-b39b-c5d924…","""citeo_5088015001""",true,"""b8b449f5-6251-583c-b39b-c5d924…"
"""ecomaison_2009874""","""refashion_TLC-REFASHION-PAV-34…",true,"""23960013-b0d2-49ea-aa7a-1a1ef6…"
"""ademesinoedecheteries_4952""","""ecodds_FD0194""",true,"""6b22d9c8-2e3a-4d2f-a104-125393…"
"""citeo_1634502001""","""citeo_1634502003""",true,"""4d9a6c31-a5f9-5e8e-9838-abb0a6…"
…,…,…,…
"""9b5caaa4-641d-572e-9833-ba878e…","""citeo_3939729001""",true,"""9b5caaa4-641d-572e-9833-ba878e…"
"""b388dbdb-3907-5899-8f44-25dcfe…","""citeo_1765490001""",true,"""b388dbdb-3907-5899-8f44-25dcfe…"
"""citeo_1692932001""","""citeo_1692932002""",true,"""6f36e26c-f368-5248-84e7-e5e974…"


In [56]:
df_pairs = pl.concat(
    [
        df_manual_labeling,
        df_negatives_via_parent_change,
        df_negatives_random.sample(
            n=(
                1000
                - len(df_manual_labeling.filter(pl.col("label").not_()))
                - len(df_negatives_via_parent_change)
            ),
            seed=42,
        ),
        df_positives.filter(
            pl.col("cluster_id").is_in(
                df_positives.select(pl.col("cluster_id").unique())
                .sample(
                    n=(1000 - len(df_manual_labeling.filter(pl.col("label")))) / 7,
                    seed=42,
                )
                .get_column("cluster_id")
                .to_list()
            )
        ),
    ]
)

In [57]:
df_pairs

identifiant_unique_i,identifiant_unique_j,label,cluster_id
str,str,bool,str
"""07e07779-388e-5971-947b-b999ae…","""lokki_67692fc641d8badbf051ec32""",true,"""7758461304653219539"""
"""085352c0-77c6-599d-b54a-02ef2a…","""772950c8-5202-5394-b0ae-8ebbda…",false,"""085352c0-77c6-599d-b54a-02ef2a…"
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5487990001""",false,"""085352c0-77c6-599d-b54a-02ef2a…"
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488009001""",false,"""085352c0-77c6-599d-b54a-02ef2a…"
"""085352c0-77c6-599d-b54a-02ef2a…","""citeo_5488015001""",false,"""085352c0-77c6-599d-b54a-02ef2a…"
…,…,…,…
"""ocab_valobat_DCT_69860_DEUX_00""","""refashion_TLC-REFASHION-PAV-34…",true,"""f0061353-c9dd-5cff-86eb-27ffed…"
"""refashion_TLC-REFASHION-PAV-32…","""refashion_TLC-REFASHION-PAV-34…",true,"""020823d8-09b6-4207-9fec-1c75a6…"
"""refashion_TLC-REFASHION-PAV-32…","""refashion_TLC-REFASHION-PAV-34…",true,"""28a640e8-0fad-5e00-aaab-212c32…"


In [53]:
df_pairs.group_by("label").len()

label,len
bool,u32
false,1000
true,1129


# Sauvegarde

In [54]:
df_pairs.write_csv(DATASETS_PATH / "dataset_20260707.csv")